# 10/30 new table with the first version of (simple) features 

Key is [ENROLID, COST_YEAR] (or [ENROLID, CLAIM_YEAR] because CLAIM_YEAR is always COST_YEAR-1)

Number of enrollees per year:

+----------+--------+
|CLAIM_YEAR|   COUNT|
+----------+--------+
|      2014|19264519|
|      2015|21330307|
|      2016|18821468|
|      2017|17311177|
|      2018|16712290|
|      2019|15418046|
|      2020|14867116|
|      2021|14470056|
|      2022|14278277|
|      2023|14568251|
+----------+--------+

## Read in data + filter for 2017 and 2018 only

In [1]:
import sys
print(f"Python version: {sys.version}")
import json
import logging
import csv
import gzip
import re
import pandas as pd
import numpy as np
from functools import reduce
from pyspark.sql.types import StringType,DecimalType,DoubleType,IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.feature import VectorAssembler, Bucketizer
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession,Row
from pyspark import SparkConf
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# -----------------------------------------------------------------------------
# INITIALIZE LOGGING
# -----------------------------------------------------------------------------
f = '%(asctime)-15s %(levelname)-8s %(message)s'
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")
logging.basicConfig(format=f)


from IPython.core.magic import register_cell_magic

# -----------------------------------------------------------------------------
# start_spark
# -----------------------------------------------------------------------------
def start_spark(
    driver_memory="100g",
    storage_fraction=0.5,
    num_nodes=10,
):
    """Initialize spark

    Arguments:
        driver_memory: Maximum heap size for the Spark driver Java
            virtual machine.
        storage_fraction: Controls what portion of Spark's unified
            memory is reserved for storage (i.e., caching/persisting data
            and broadcast variables), as a fraction of the total
            execution + storage memory pool.
            If you cache/persist a lot of data, and you're evicting
            data too early, you might increase this value (e.g. 0.6 or 0.7).
            Conversely, if your job is shuffle-heavy and fails due to
            memory pressure, you might decrease it (e.g. 0.3).
        num_nodes: How many concurrent threads to use while running
            in "local mode" (i.e. in a single machine instead of a cluster).
            Use '*' to use all cores, or an integer > 0 for a specific
            number of threads.
    """

    conf = SparkConf().setAppName("My_Application")
    conf.set("spark.driver.memory", driver_memory)
    conf.set("spark.memory.storageFraction", str(storage_fraction))
    conf.setMaster(f"local[{num_nodes}]")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel('WARN')

    return spark


Python version: 3.11.0 (main, Jun 13 2025, 14:48:45) [Clang 16.0.0 (clang-1600.0.26.6)]


In [2]:
spark = start_spark(num_nodes=10)

@register_cell_magic
def spark_sql(line, cell):
    result = spark.sql(cell)
    result.show(n=1000)

# -- READ ENROLLMENT AND DATA TABLES
enrollment_file = f"/Users/charles/DATA/marketscan/data/normalized/features_v1"
logger.info(f">>> Reading enrollment file: {enrollment_file}")
df_enrollment = spark.read.format("parquet").load(enrollment_file)
df_enrollment.createOrReplaceTempView('enrollment')
logger.info(f">>> ENROLLMENT has {df_enrollment.count():,} rows")
logger.info(f">>> ENROLLMENT has {df_enrollment.select('ENROLID').distinct().count():,} unique enrollees")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/01 08:50:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2025-11-01 08:50:20,283 INFO     >>> Reading enrollment file: /Users/charles/DATA/marketscan/data/normalized/features_v1
25/11/01 08:50:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2025-11-01 08:50:22,058 INFO     >>> ENROLLMENT has 167,041,507 rows
2025-11-01 08:50:48,458 INFO     >>> ENROLLMENT has 55,868,722 unique enrollees 


In [4]:

spark_2018 = (
    df_enrollment
     .filter(F.col("COST_YEAR")== 2018)
     .filter(F.col("cluster_0")>0)
)
df_2018 = spark_2018.toPandas()
df_2018.head()

,ENROLID,EFAMID,COST_YEAR,CLAIM_YEAR,SEX,AGE,AGEGRP,AGE_BUCKET,EMPREL,COST_MEMDAYS,...,cluster_0,cluster_1,cluster_2,cluster_3,cluster_4,cluster_5,cluster_6,cluster_7,cluster_8,cluster_9
0,27128501,271285,2018,2017,2,56,5,7,2,307,...,605.00,300.64,0.0,0.0,33817.21,0.00,19588.57,0.0,394.27,19952.23
1,32772005,327720,2018,2017,2,51,4,6,2,131,...,322.14,0.00,0.0,0.0,342.22,0.00,1924.13,0.0,318.76,523.90
2,128498602,1284986,2018,2017,2,60,5,7,2,365,...,651.17,0.00,0.0,0.0,0.00,1457.96,1118.51,0.0,541.67,0.00
3,129909401,1299094,2018,2017,2,59,5,7,1,365,...,160.02,0.00,0.0,0.0,0.00,0.00,911.90,0.0,945.72,0.00
4,131330202,1313302,2018,2017,2,61,5,7,2,334,...,1316.00,10.14,0.0,0.0,1850.05,0.00,3901.66,0.0,129.86,3114.60


In [9]:
print(df_2018.SEX.value_counts(normalize=True))
print(df_2018.AGEGRP.value_counts(normalize=True))
print(df_2018.REGION.value_counts(normalize=True))
print(df_2018.MSA_NEW.value_counts(normalize=True))
print(df_2018.EGEOLOC.value_counts(normalize=True))
print(df_2018.EMPREL.value_counts(normalize=True))



SEX
2    0.529688
1    0.470312
Name: proportion, dtype: float64
AGEGRP
5    0.411005
4    0.244008
3    0.132009
2    0.128839
1    0.065397
6    0.018742
Name: proportion, dtype: float64
REGION
3    0.437339
2    0.224021
1    0.196683
4    0.139462
5    0.002496
Name: proportion, dtype: float64
MSA_NEW
0.0        0.127691
35620.0    0.073746
19100.0    0.030953
37980.0    0.025057
12060.0    0.023412
             ...   
27980.0    0.000038
20940.0    0.000038
41980.0    0.000026
10380.0    0.000013
41900.0    0.000004
Name: proportion, Length: 393, dtype: float64
EGEOLOC
01    0.097684
12    0.084634
33    0.078219
49    0.076962
62    0.054051
18    0.051889
19    0.051241
13    0.035139
11    0.035063
16    0.032011
34    0.031701
36    0.027898
25    0.023009
38    0.022010
17    0.019507
65    0.019139
42    0.018056
20    0.017481
35    0.017314
41    0.016352
06    0.016280
52    0.014769
53    0.014663
44    0.014591
37    0.012285
47    0.011452
43    0.010619
04    0.010531

In [3]:
"""It has the following columns:

ENROLID: Enrollee ID
EFAMID: Enrollee's Family ID
COST_YEAR: Target year (we want to predict the cost for this year)
CLAIM_YEAR: Observation year (use claims data from this year). Always equal to "COST_YEAR-1"
SEX: 1=male, 2=female
AGE: Age in year
AGEGRP: Marketscan's age group
AGE_BUCKET: Our own age group
EMPREL: Relation to employee (1=Employee, 2=Spouse, 3=Child/dependent)
COST_MEMDAYS: total membership/enrollment days during COST_YEAR
CLAIM_MEMDAYS: total membership/enrollment days during CLAIM_YEAR
RX: Rx flag (this person had Rx data during CLAIM_YEAR)
MSA: MSA of residence (during CLAIM_YEAR)
MSA_NEW: MSA of residence (during CLAIM_YEAR) - fixed by us
EGEOLOC: Geographic location (during CLAIM_YEAR)
REGION: Marketscan's region (during CLAIM_YEAR)
INCOME_LEVEL: Median income level of MSA (during CLAIM_YEAR)
INDSTRY: Employer's industry
groupid: Our own group id (practically useless nowdays...)
PAY_TOTAL_ADJ: Total cost during COST_YEAR (adjusted for inflation)
NETPAY_TOTAL_ADJ: Total plan cost during COST_YEAR (adjusted for inflation)
PREV_PAY_TOTAL_ADJ: Total cost during CLAIM_YEAR (adjusted for inflation)
cluster_0: Total cost (during CLAIM_YEAR) for ICD codes in cluster #0
cluster_1: Total cost (during CLAIM_YEAR) for ICD codes in cluster #1
cluster_2: Total cost (during CLAIM_YEAR) for ICD codes in cluster #2
cluster_3: Total cost (during CLAIM_YEAR) for ICD codes in cluster #3
cluster_4: Total cost (during CLAIM_YEAR) for ICD codes in cluster #4
cluster_5: Total cost (during CLAIM_YEAR) for ICD codes in cluster #5
cluster_6: Total cost (during CLAIM_YEAR) for ICD codes in cluster #6
cluster_7: Total cost (during CLAIM_YEAR) for ICD codes in cluster #7
cluster_8: Total cost (during CLAIM_YEAR) for ICD codes in cluster #8
cluster_9: Total cost (during CLAIM_YEAR) for ICD codes in cluster #9"""


'It has the following columns:\n\nENROLID: Enrollee ID\nEFAMID: Enrollee\'s Family ID\nCOST_YEAR: Target year (we want to predict the cost for this year)\nCLAIM_YEAR: Observation year (use claims data from this year). Always equal to "COST_YEAR-1"\nSEX: 1=male, 2=female\nAGE: Age in year\nAGEGRP: Marketscan\'s age group\nAGE_BUCKET: Our own age group\nEMPREL: Relation to employee (1=Employee, 2=Spouse, 3=Child/dependent)\nCOST_MEMDAYS: total membership/enrollment days during COST_YEAR\nCLAIM_MEMDAYS: total membership/enrollment days during CLAIM_YEAR\nRX: Rx flag (this person had Rx data during CLAIM_YEAR)\nMSA: MSA of residence (during CLAIM_YEAR)\nMSA_NEW: MSA of residence (during CLAIM_YEAR) - fixed by us\nEGEOLOC: Geographic location (during CLAIM_YEAR)\nREGION: Marketscan\'s region (during CLAIM_YEAR)\nINCOME_LEVEL: Median income level of MSA (during CLAIM_YEAR)\nINDSTRY: Employer\'s industry\ngroupid: Our own group id (practically useless nowdays...)\nPAY_TOTAL_ADJ: Total cost du